In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from urllib.request import urlopen

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
IS_DATALORE = any(k.startswith("DATALORE") for k in os.environ) or os.path.exists("/data/notebook_files")
IS_DEEPNOTE = any(k.startswith("DEEPNOTE") for k in os.environ) or os.path.exists("/work")
IS_CLOUD = IS_COLAB or IS_KAGGLE or IS_DATALORE or IS_DEEPNOTE

PYPROJECT_URL = "https://raw.githubusercontent.com/dramirezbe/notebook-toolkit-gcpds/main/pyproject.toml"

# Parse pinned versions from pyproject.toml
toml = urlopen(PYPROJECT_URL).read().decode()
deps = []
in_deps = False
for line in toml.splitlines():
    if line.strip().startswith("dependencies"):
        in_deps = True
        continue
    if in_deps:
        if line.strip().startswith("]"):
            break
        dep = line.strip().strip(",").strip('"')
        if dep:
            deps.append(dep)

TARGET = {d.split("==")[0]: d.split("==")[1] for d in deps if "==" in d}

def check_versions():
    for pkg, expected in TARGET.items():
        try:
            if version(pkg) != expected:
                return False
        except PackageNotFoundError:
            return False
    return True

if not check_versions():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    uv_args = [sys.executable, "-m", "uv", "pip", "install", "--system", "--force-reinstall"]
    if IS_DATALORE:
        uv_args.append("--break-system-packages")
    subprocess.run([
        *uv_args,
        "notebook-toolkit-gcpds @ git+https://github.com/dramirezbe/notebook-toolkit-gcpds.git",
        *deps,
    ], check=True)

    env = "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Datalore" if IS_DATALORE else "DeepNote" if IS_DEEPNOTE else "Local"
    print(f"Environment: {env} | Python: {sys.version.split()[0]}")
    print("Packages installed. Restarting kernel...")
    try:
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("Restart the kernel manually: Runtime > Restart session")
else:
    print("Packages already installed.")

In [ ]:
import numpy
import pandas
import scipy
import sklearn

print(f"NumPy:       {numpy.__version__}")
print(f"Pandas:      {pandas.__version__}")
print(f"SciPy:       {scipy.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")